In [1]:
%pip install -U langchain langchain-core "langchain-community<=0.4" langchain-anthropic rapidfuzz langchain-experimental langchain-text-splitters langchain-neo4j langchain-huggingface langchain-ollama neo4j python-dotenv sentence-transformers ragas

  Using cached langchain_community-0.4-py3-none-any.whl.metadata (3.0 kB)
  Using cached rapidfuzz-3.14.5-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (12 kB)
  Using cached langchain_huggingface-1.2.2-py3-none-any.whl.metadata (4.0 kB)
  Using cached sentence_transformers-5.6.0-py3-none-any.whl.metadata (18 kB)
  Using cached ragas-0.4.3-py3-none-any.whl.metadata (23 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached marshmallow-3.26.2-py3-none-any.whl.metadata (7.3 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached mypy_extensions-1.1.0-py3-none-any.whl.metadata (1.1 kB)
INFO: pip is looking at multiple versions of langchain-experimental to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_experimental-0.4.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached langchain_experimental-0.4.1-py3-none-any.whl.metadata (1.3 kB)


In [2]:
from dotenv import load_dotenv
import os
import json

from copy import deepcopy


from pydantic import BaseModel, Field
from neo4j import GraphDatabase, Driver

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_neo4j import Neo4jGraph, Neo4jVector
from langchain_neo4j.vectorstores.neo4j_vector import remove_lucene_chars

from langchain_experimental.graph_transformers import LLMGraphTransformer

from langchain_community.graphs.graph_document import GraphDocument
from langchain_core.documents import Document
from langchain_community.graphs.graph_document import Node, Relationship

from typing import List, Dict, Any

from pydantic import BaseModel, Field, ValidationError
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_community.graphs.graph_document import Node, Relationship, GraphDocument

from ragas import EvaluationDataset

from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

load_dotenv()

True

In [3]:
from langchain_anthropic import ChatAnthropic

cloud_llm = ChatAnthropic(
    model_name="claude-haiku-4-5-20251001",
    temperature=0,
    api_key=os.getenv("ANTHROPIC_API_KEY"),
)

In [34]:
graph = Neo4jGraph(database="cloud4")

In [5]:
# def serialize_node(node):
#     return Node(id=node["id"], type=node["type"], properties=node["properties"])


# def serialize_relationship(relation, nodes: list[Node]):
#     source = Node(id=relation["source"]["id"], type=relation["source"]["type"], properties=relation["source"]["properties"])
#     target = Node(id=relation["target"]["id"], type=relation["target"]["type"], properties=relation["target"]["properties"])
#     return Relationship(
#         source=source,
#         target=target,
#         type=relation["type"],
#         properties=relation["properties"],
#     )

# def cast_to_graph_document(json_object:any):
#     nodes: list[Node] = []
#     for node in json_object["nodes"]:
#         nodes.append(serialize_node(node))

#     relationships: list[Relationship] = []
#     for rel in json_object["relationships"]:
#         relationships.append(serialize_relationship(rel, nodes))
    
#     source: Document = Document(metadata=json_object["document"]["metadata"], page_content=json_object["document"]["page_content"])
#     return GraphDocument(nodes=nodes, relationships=relationships, source=source)


def convert_to_graph_doc(s: str):
    graph_documents = eval(s, {
        "GraphDocument": GraphDocument,
        "Document": Document,
        "Node": Node,
        "Relationship": Relationship,
    })
    return graph_documents

In [6]:
# class NodeModel(BaseModel):
#     id: str
#     type: str
#     properties: Dict[str, Any] = Field(default_factory=dict)


# class RelationshipModel(BaseModel):
#     source: NodeModel
#     target: NodeModel
#     type: str
#     properties: Dict[str, Any] = Field(default_factory=dict)


# class DocumentModel(BaseModel):
#     metadata: Dict[str, Any] = Field(default_factory=dict)
#     page_content: str


class ExtractionResult(BaseModel):
    nodes: List[Node] = Field(default_factory=list)
    relationships: List[Relationship] = Field(default_factory=list)
    document: Document

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a high-recall literary information extraction system for building a knowledge graph from a single source document.

Your task:
Extract all important entities, attributes, and relationships explicitly stated in the input document, and return exactly one valid JSON object with the keys:
- nodes
- relationships
- document

Primary objective:
Maximize useful recall for downstream question answering over a literary corpus.
Prefer extracting too many relevant explicit facts over omitting important ones.

Hard rules:
- Output only valid JSON.
- Do not output markdown, code fences, commentary, or explanations.
- Do not output Python objects such as GraphDocument(...).
- Use only facts explicitly stated in the input document.
- Do not invent facts.
- Do not use background knowledge from outside the document.
- Process the entire input document as one unit.
- If no entities or relationships are present, return empty lists.

Strict schema:
- Every node.type must be exactly one of:
  Person, Organization, Place, Document, Artifact, Event, Case, Role
- Every relationship.type must be exactly one of:
  WORKS_FOR, HAS_ROLE, ALIAS_OF, AFFILIATED_WITH, LOCATED_IN, RESIDES_AT, PARTICIPATED_IN, OCCURRED_AT, OCCURRED_ON, CREATED, ADDRESSED_TO, POSSESSES, TARGETS, RELATED_TO_CASE
- Do not use any other node types or relationship types.

Critical property constraint:
- The properties field of every node and relationship must be a flat JSON object.
- Allowed property values are only:
  - string
  - number
  - boolean
  - list of strings
  - list of numbers
  - list of booleans
- Do NOT use nested objects, nested dictionaries, maps, or lists of objects inside properties.
- If a detail would naturally be structured as an object, flatten it into one or more primitive properties instead.
- Example: use "hair_color": "black" and "face_description": "pale, refined" instead of nested appearance objects.
- Example: use "surface_forms": ["Holmes", "Mr. Holmes"] instead of nested alias objects.

What to extract:
Extract all explicit, question-relevant information, including:

1. Persons
- named characters
- partially named characters if no fuller name appears
- unnamed but clearly important referential persons, e.g. "the landlady", "the old man", "the bride", "the doctor"

2. Places
- addresses, rooms, buildings, streets, cities, regions, countries
- residences and places visited

3. Organizations and groups
- families, houses, institutions, police, royal houses, clubs, employers, professions where relevant

4. Important objects and documents
- letters, notes, photographs, weapons, tools, clothing, disguises, jewelry, keys, furniture, animals, vehicles, etc.
- include plot-relevant objects even if not named formally

5. Events and actions
- meetings, departures, marriages, threats, discoveries, thefts, disguises, conversations, journeys, attacks, investigations, requests, observations
- create event nodes when the event itself is important for later reasoning or connects multiple entities

6. Descriptive attributes
Store important explicit attributes in node.properties, especially:
- physical appearance: age, approximate age, hair, beard, eyes, face, complexion, scars, build, height, clothing, posture, expression
- identity and naming: titles, aliases, alternate names, epithets, descriptions
- social and personal attributes: occupation, profession, rank, family role, marital status, nationality, social status
- emotional or mental state: frightened, agitated, calm, angry, drunk, ill, tired, etc.
- location or residence
- possessions or associated objects
- distinguishing features useful for retrieval or disambiguation
- any other explicit traits likely to matter for question answering

Coreference and canonicalization:
- Resolve pronouns and shortened mentions to the most complete explicit name that appears in the same document.
- Examples:
  - "Holmes", "Mr. Holmes", "he" -> "Sherlock Holmes" if "Sherlock Holmes" appears in the document
  - "Watson", "I", "the narrator" -> "Dr. Watson" if "Dr. Watson" appears in the document
  - "the woman" -> "Irene Adler" only if "Irene Adler" explicitly appears in the same document and the reference is clearly unambiguous
- If the fuller canonical name does not appear in the document, keep the explicit local mention as its own node.
- Do not merge uncertain identities.
- Prefer preserving a distinct explicit entity over making a wrong merge.

Allowed node schemas:
- Person:
  - id: canonical human-readable name
  - type: "Person"
  - properties may include: canonical_name, surface_forms, gender, nationality, description, age, occupation, role, title, marital_status, residence, appearance, hair_color, beard, eye_description, face_description, complexion, build, height_description, scar_description, posture, expression, emotional_state, aliases
- Organization:
  - properties may include: name, org_type, surface_forms
- Place:
  - properties may include: name, place_type, surface_forms
- Document:
  - properties may include: title_or_label, doc_type, quoted_text, date_text, language
- Artifact:
  - properties may include: name, artifact_type, description
- Event:
  - properties may include: event_type, summary, date_text, sequence, certainty
- Case:
  - properties may include: case_name, summary, status
- Role:
  - properties may include: role_name, role_type, temporal_scope

Node requirements:
- Each node must have:
  - id: canonical human-readable identifier, preferably the most complete explicit form in the document
  - type: exactly one of the allowed node types above
  - properties: flat JSON object with primitive values only

Relationship requirements:
- Extract all important explicit relationships between nodes.
- Relationship types are restricted to the allowed list above.
- Each relationship must have:
  - source: node object
  - target: node object
  - type: exactly one of the allowed relationship types
  - properties: flat JSON object with primitive values only

Event extraction guidance:
- Create event nodes only when useful, e.g. a marriage, departure, attack, discovery, consultation, or theft.
- Link participants to events with relations such as PARTICIPATED_IN, OCCURRED_AT, OCCURRED_ON, CREATED, ADDRESSED_TO, POSSESSES, TARGETS, RELATED_TO_CASE.
- Do not create trivial event nodes for every sentence.

Deduplication:
- Each real-world entity should appear only once per document after clear coreference resolution.
- Do not create duplicate nodes that differ only by shortened mentions when the fuller canonical mention is explicit and unambiguous in the same document.

Document object:
- Return the input document unchanged under the key "document".
- Preserve its metadata and page_content.

Output format:
Return exactly one JSON object with exactly these top-level keys:
- nodes
- relationships
- document
"""
    ),
    (
        "human",
        "Input document:\n{input_document}"
    )
])


# llm = ChatOllama(
#     model="qwen3.6:latest",
#     base_url="http://192.168.178.67:11434",
#     temperature=0,
#     # format=ExtractionResult.model_json_schema(),
#     reasoning=False,
#     keep_alive="24h",
#     num_predict=-1,
#     format="json"
# )

llm_json = cloud_llm.with_structured_output(ExtractionResult, include_raw=True)


json_chain = prompt | llm_json

# response = json_chain.invoke({"input_document":json.dumps(
#                 {
#                     "metadata": documents[2].metadata,
#                     "page_content": documents[2].page_content,
#                 },
#                 ensure_ascii=False,
#             )})

In [8]:
from copy import deepcopy
from typing import Dict, Tuple
from langchain_community.graphs.graph_document import GraphDocument, Node, Relationship

def _normalize_id(node_id: str, alias_map: Dict[str, str]) -> str:
    key = node_id.strip().lower()
    return alias_map.get(key, node_id.strip())

def _merge_properties(a: dict, b: dict) -> dict:
    merged = deepcopy(a) if a else {}
    for k, v in (b or {}).items():
        if k not in merged:
            merged[k] = v
        else:
            if merged[k] == v:
                continue
            if isinstance(merged[k], list):
                existing = merged[k]
            else:
                existing = [merged[k]]
            if isinstance(v, list):
                for item in v:
                    if item not in existing:
                        existing.append(item)
            else:
                if v not in existing:
                    existing.append(v)
            merged[k] = existing
    return merged

def _canonical_node(node: Node, alias_map: Dict[str, str]) -> Node:
    new_node = deepcopy(node)
    new_node.id = _normalize_id(str(node.id), alias_map)
    return new_node

def resolve_results(validated_result: GraphDocument) -> GraphDocument:
    alias_map = {
        "holmes": "Sherlock Holmes",
        "sherlock": "Sherlock Holmes",
        "mr. holmes": "Sherlock Holmes",
        "watson": "Dr. John H. Watson",
        "dr. watson": "Dr. John H. Watson",
        "the narrator": "Dr. John H. Watson",
        "narrator": "Dr. John H. Watson",
        "the woman": "Irene Adler",
        "miss adler": "Irene Adler",
        "irene norton": "Irene Adler",
        "i": "Dr. John H. Watson",
        "he": "Sherlock Holmes"
    }

    canonical_nodes: Dict[Tuple[str, str], Node] = {}

    for node in validated_result.nodes:
        cnode = _canonical_node(node, alias_map)
        key = (str(cnode.id), str(cnode.type))
        if key not in canonical_nodes:
            canonical_nodes[key] = cnode
        else:
            canonical_nodes[key].properties = _merge_properties(
                canonical_nodes[key].properties,
                cnode.properties
            )

    resolved_relationships = []

    for rel in validated_result.relationships:
        new_rel = deepcopy(rel)
        new_rel.source = _canonical_node(rel.source, alias_map)
        new_rel.target = _canonical_node(rel.target, alias_map)

        source_key = (str(new_rel.source.id), str(new_rel.source.type))
        target_key = (str(new_rel.target.id), str(new_rel.target.type))

        if source_key in canonical_nodes:
            new_rel.source = canonical_nodes[source_key]
        else:
            canonical_nodes[source_key] = new_rel.source

        if target_key in canonical_nodes:
            new_rel.target = canonical_nodes[target_key]
        else:
            canonical_nodes[target_key] = new_rel.target

        resolved_relationships.append(new_rel)

    deduped_relationships = []
    seen_rels = set()

    for rel in resolved_relationships:
        rel_key = (
            str(rel.source.id),
            str(rel.source.type),
            str(rel.type),
            str(rel.target.id),
            str(rel.target.type),
            json.dumps(rel.properties, sort_keys=True, ensure_ascii=False),
        )
        if rel_key not in seen_rels:
            seen_rels.add(rel_key)
            deduped_relationships.append(rel)

    return GraphDocument(
        nodes=list(canonical_nodes.values()),
        relationships=deduped_relationships,
        source=validated_result.source
    )

In [9]:
def extract_graph_document(resolved: ExtractionResult):
    return GraphDocument(nodes=resolved.nodes, relationships=resolved.relationships, source=resolved.document)

# def resolve_results(validated_result: GraphDocument):
#     resolved_nodes = []
#     resolved_relationships = []
#     alias_map = {
#         "holmes": "Sherlock Holmes",
#         "sherlock": "Sherlock Holmes",
#         "watson": "Dr. Watson",
#         "the woman": "Irene Adler",
#         "miss adler": "Irene Adler",
#         "mr. holmes": "Sherlock Holmes",
#         "i": "Dr. Watson",
#         "narrator": "Dr. Watson",
#         "he": "Sherlock Holmes"
#     }
#     for node in validated_result.nodes:
#         new_node = deepcopy(node)
#         if node.id.lower() in alias_map:
#             new_node.id = alias_map[node.id.lower()]
#             resolved_nodes.append(new_node)

#     for rel in validated_result.relationships:
#         source_node = rel.source
#         target_node = rel.target
#         new_rel = deepcopy(rel)
#         if source_node.id.lower() in alias_map:
#             new_node = deepcopy(source_node)
#             new_node.id = alias_map[source_node.id.lower()]
#             new_rel.source = new_node
#         if target_node.id.lower() in alias_map:
#             new_node = deepcopy(target_node)
#             new_node.id = alias_map[target_node.id.lower()]
#             new_rel.target = new_node
        
#         resolved_relationships.append(new_rel)
    
#     return GraphDocument(nodes=resolved_nodes, relationships=resolved_relationships, source=validated_result.source)


def extract_graph_document_json_enforced(doc: Document) -> GraphDocument:
    response = json_chain.invoke({"input_document": doc})
    # print(response)
    # json_data = json.loads(response.content)
    # print(f"json: {json_data}")

    validated = ExtractionResult.model_validate(response["parsed"])
    graph_doc = extract_graph_document(validated)
    return resolve_results(graph_doc)


def process_documents_json_enforced(documents: List[Document]) -> List[GraphDocument]:
    graph_docs = []
    for i, doc in enumerate(documents):
        try:
            graph_doc = extract_graph_document_json_enforced(doc)
            graph_docs.append(graph_doc)
            print(f"Processed document {i+1}/{len(documents)}")
        except (ValidationError, json.JSONDecodeError, Exception) as e:
            print(f"Failed document {i+1}: {e}")
    return graph_docs



In [ ]:
# graph_docs = []
# for doc in documents[:5]:

# directory = "../data/chapters/"
# for file in os.listdir(directory):
#     print(f"processing {file}...")
loader = TextLoader(file_path=f"../data/chapters/chapter_1.txt")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=150)
documents = text_splitter.split_documents(documents=docs)

graph_docs = process_documents_json_enforced(documents[:15])

with open(f"../data/local_2/chapter_1_15.txt", "w") as file_out:
    file_out.write(str(graph_docs))

In [ ]:
# import os
# import json
# from collections.abc import Mapping

# def flatten_dict(d, parent_key="", sep="_"):
#     items = {}
#     for k, v in d.items():
#         new_key = f"{parent_key}{sep}{k}" if parent_key else k
#         if isinstance(v, Mapping):
#             items.update(flatten_dict(v, new_key, sep=sep))
#         elif isinstance(v, list):
#             cleaned = []
#             for item in v:
#                 if isinstance(item, Mapping):
#                     cleaned.append(json.dumps(item, ensure_ascii=False))
#                 else:
#                     cleaned.append(item)
#             items[new_key] = cleaned
#         else:
#             items[new_key] = v
#     return items

# def sanitize_graph_document(graph_doc):
#     for node in graph_doc.nodes:
#         if node.properties:
#             node.properties = flatten_dict(node.properties)
#     for rel in graph_doc.relationships:
#         if rel.properties:
#             rel.properties = flatten_dict(rel.properties)
#     return graph_doc

directory = "../data/cloud/"
sanitized_docs = []
for filename in os.listdir(directory):
    with open(os.path.join(directory,filename), "r", encoding="utf-8") as file_in:
        string_graph_docs = file_in.read()

    parsed_graph_document = convert_to_graph_doc(string_graph_docs)
    for graph_doc in parsed_graph_document:
        sanitized_docs.append(resolve_results(graph_doc))

    # sanitized_docs.extend([sanitize_graph_document(doc) for doc in parsed_graph_document])

    # graph.add_graph_documents(
    #     parsed_graph_document,
    #     baseEntityLabel=True,
    #     include_source=True,
    # )
    # print(f"Added {filename} to graph.")

Added chapter_1.txt to graph.


In [38]:
len(parsed_graph_document)

53

In [11]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en",
    model_kwargs = {"device": "cpu"}
)

vector_index = Neo4jVector.from_existing_graph(
    embeddings,
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
    database="cloud4"
)
def invoke_vector_retriever(query:str, k:int=10):
    vector_retriever = vector_index.as_retriever(search_kwargs={"k": k})
    return vector_retriever.invoke(query)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [12]:
driver = GraphDatabase.driver(
        uri = os.environ["NEO4J_URI"],
        auth = (os.environ["NEO4J_USERNAME"],
                os.environ["NEO4J_PASSWORD"]))

def create_fulltext_index(tx):
    query = '''
    CREATE FULLTEXT INDEX `fulltext_entity_id` 
    FOR (n:__Entity__) 
    ON EACH [n.id];
    '''
    tx.run(query)

# Function to execute the query
def create_index():
    with driver.session(database="cloud4") as session:
        session.execute_write(create_fulltext_index)
        print("Fulltext index created successfully.")

# Call the function to create the index
try:
    create_index()
except:
    print("Index creation failed")
    pass

# Close the driver connection
driver.close()

Fulltext index created successfully.


In [16]:
class Entities(BaseModel):
    """Entity mentions relevant for graph retrieval."""
    names: list[str] = Field(
        ...,
        description="All entity names explicitly mentioned in the question that may appear as graph nodes."
    )

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You extract graph entity mentions from questions. Return all explicitly mentioned entities that may exist as graph nodes."
    ),
    (
        "human",
        "Extract entities from this question: {question}"
    ),
])


entity_chain = prompt | cloud_llm.with_structured_output(Entities, method="function_calling")

In [17]:
entity_chain.invoke("How are Sherlock Holmes and Irene Adler related?")

Entities(names=['Sherlock Holmes', 'Irene Adler'])

In [19]:
def generate_full_text_query(input: str) -> str:
    words = [el for el in remove_lucene_chars(input).split() if el]
    if not words:
        return ""
    full_text_query = " AND ".join([f"{word}~2" for word in words])
    print(f"Generated Query: {full_text_query}")
    return full_text_query.strip()


def graph_retriever(question: str, k: int = 10) -> str:
    result = []
    entities = entity_chain.invoke(question)
    if not entities:
        raise Exception("No entities present")

    for entity in entities.names:
        response = graph.query(
            """
        CALL db.index.fulltext.queryNodes('fulltext_entity_id', $query, {limit: 2})
        YIELD node, score
        CALL {
          WITH node
          MATCH (node)-[r:!MENTIONS]->(neighbor)
          RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output
          UNION
          WITH node
          MATCH (node)<-[r]-(neighbor)
          RETURN neighbor.id + ' - ' + type(r) + ' -> ' + node.id AS output
        }
        RETURN output
        LIMIT 50
        """,
            {"query": generate_full_text_query(entity)},
        )

        result.extend(row["output"] for row in response)

    return "\n".join(sorted(set(result))[:k])

In [ ]:
graph_retriever("Irene Adler")